# Week 5 

## Problem 1

### Task 1

$$
(u_i - u)^2 = u_i^2 -2u u_i +u^2
$$

We can then formulate it as a quadratic probblem as: 

$$
H = I
$$

$$
g^T = (-2u, \dots, -2u)
$$

$$
\gamma = u^2

$$

$$
b = (-1, 0, 0, ..., 0)^T
$$

$$
A^T = 
\begin{bmatrix}
-1 & 0 & 0 & ... & 1 & 0 \\
1 & -1 & 0 & ...& 0& 0 \\
0 & 1 & -1 & ...& 0 & 0 \\
... \\
\end{bmatrix}

$$

### Task 2

We let $n=10$, $u = 0.2$ and $d_0 = 1$

The larange function is:

$$
L(x, \lambda) = x^T I x + g^Tx + \gamma + \lambda_1 c_1(x) + \sum_{i = 1}^{8} \lambda_{2,i} c_{2,i}(x) + \lambda_3c_3(x)
$$

where 
$$
c_1(x) = -u_1 +u_8 + 1 
$$
$$
c_{2,i} = u_i - u_{i + 1} = 0 \forall i = 1, 2, ..., 8
$$
$$
c_3(x) = u_{7} - u_{8} - u_{9} = 0
$$

We can then formulate the first order optimality conditions as: 


Becaue the hessian is the identity matrix, then we know its positive definite and therefore it's a stricty convex problem. 

Due to only handling equality constraints, we know now that the first order KKT conditions is nescecarry to ensure a global minimizer. 


### Task 3

In [79]:
import numpy as np 

def construct_matrix(n, u, d0):
    """Constructs H, g, A, b for the equality-constrained QP."""
    N = n + 1
    H = np.eye(N)
    g = np.ones(N) * (-2) * u
    
    # Number of equality constraints is N-1
    b = np.zeros(N - 1)
    b[0] = -d0
    
    A = np.zeros((N - 1, N))
    A[0, 0] = -1
    A[0, -2] = 1

    for i in range(1, n - 1):
        A[i, i - 1] = 1
        A[i, i] = -1

    A[-1, n - 2] = 1
    A[-1, n - 1] = -1
    A[-1, n] = -1

    return H, g, A.T, b


n = 10000
u = 0.2
d0 = 1

H, g, A, b = construct_matrix(n, u, d0)
A.T

array([[-1.,  0.,  0., ...,  0.,  1.,  0.],
       [ 1., -1.,  0., ...,  0.,  0.,  0.],
       [ 0.,  1., -1., ...,  0.,  0.,  0.],
       ...,
       [ 0.,  0.,  0., ...,  0.,  0.,  0.],
       [ 0.,  0.,  0., ..., -1.,  0.,  0.],
       [ 0.,  0.,  0., ...,  1., -1., -1.]], shape=(10000, 10001))

### Task 4

In [80]:
def construct_kkt_matrix(n, u, d0): 
    H, g, A, b = construct_matrix(n, u, d0) 
    m = A.shape[1]  # number of constraints
    
    KKT = np.block([[H, -A],
                    [-A.T, np.zeros((m, m))]])
    return KKT, -np.concatenate([g, b])


KKT, rhs = construct_kkt_matrix(n, u, d0)
KKT.shape, rhs.shape

((20001, 20001), (20001,))

### Task 5

In [82]:
import numpy as np
from scipy.linalg import lu

def lu_solve(KKT, b): 
    P, L, U = lu(KKT)
    y = np.linalg.solve(L, P.T @ b)
    x = np.linalg.solve(U, y)
    return x

n = 1000
u = 0.2
d0 = 1


KKT, b = construct_kkt_matrix(n, u, d0)

x = lu_solve(KKT, b)
x

array([ 0.401,  0.401,  0.401, ..., -0.602, -0.601, -0.6  ], shape=(2001,))

## Task 6

In [83]:
from scipy.linalg import ldl 
def ldl_solve(KKT, b):  
    L, D, perm = ldl(KKT) 
    bp = b[perm]
    z1 = np.linalg.solve(L, bp) 
    z2 = np.linalg.solve(D, z1) 
    x = np.linalg.solve(L.T, z2)
    return x[perm]




x = ldl_solve(KKT, b)
x

array([ 0.401,  0.401,  0.401, ..., -0.602, -0.601, -0.6  ], shape=(2001,))

## Task 7

In [85]:
from scipy.linalg import qr, null_space, solve_triangular
H, g, A, b = construct_matrix(n, u, d0)  

def null_space_solver(A, H , g, b): 
    Q1, R = qr(A, mode="economic") 
    Q2 = null_space(A.T) 

    x_y = solve_triangular(R.T, b, lower=True)

    mvec = H@Q1@x_y + g

    x_z = np.linalg.solve(Q2.T@H@Q2, -Q2.T@mvec)

    x = Q1@x_y + Q2@x_z 

    lam = np.linalg.solve(R, Q1.T @ (H @ x + g))
    return x, lam



x, lam = null_space_solver(A, H , g, b)
x

array([ 0.401,  0.401,  0.401, ...,  0.401, -0.599,  1.   ], shape=(1001,))

### Task 8